# מטלת סיום – חלק 2 | Movie Rating Prediction

**שמות חברי הצוות + ת.ז.:** <FILL IN>

**קישור ל-GitHub:** <FILL IN>

In [ ]:
# ============================================================
#  מטלת סיום – חלק 2 | Movie Rating Prediction
#  Chen Hajaj · Ariel University · Machine Learning
# ============================================================
# שמות חברי הצוות + ת.ז.:  <FILL IN>
# קישור ל-GitHub:            <FILL IN>
# ============================================================

## 1. ייבוא ספריות

In [1]:
import re
import ast
import bisect
import warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from sklearn.linear_model    import ElasticNet
from sklearn.ensemble        import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler
from sklearn.impute          import SimpleImputer
from sklearn.compose         import ColumnTransformer
from sklearn.metrics         import mean_squared_error, mean_absolute_error, r2_score
from sklearn.base            import BaseEstimator, TransformerMixin
from sklearn.model_selection import GridSearchCV

## 2. טעינת הדאטה

In [2]:
dataset = pd.read_csv("dataset.csv", na_values=['\\N', 'N/A', 'null'], low_memory=False)
# טעינת קובץ של הבמאים 
df_crew = pd.read_csv("title.crew.tsv.gz", sep='\t', usecols=['tconst', 'directors'], low_memory=False)
# מיזוג של הקבצים לדאטה שכוללת את הבמאים 
dataset = pd.merge(dataset, df_crew, on='tconst', how='left')

## 3. בניית פונקציות עזר לניקוי הנתונים 


In [3]:
# ── פונקציות ניקוי ───────────────────────────────────────────
 
def _clean_year(val):
    """ערך תקין: שנה בטווח הגיוני, מטפל גם במספרים עשרוניים כמו 1993.0"""
    if pd.isna(val):
        return np.nan
    try:
        # המרה ישירה ל-float ואז ל-int כדי להפטר מה-.0 בצורה נקייה
        year = int(float(val))
        return year
    except (ValueError, TypeError):
        return np.nan
 
 
def _clean_runtime(val):
    """ערך תקין: מספר בלבד (לא כחלק מטקסט)."""
    if pd.isna(val):
        return np.nan
    try:
        return float(val)
    except (ValueError, TypeError):
        return np.nan
 
 
def _clean_rating(val):
    """ערך תקין: מספר בין 1.0 ל-10.0."""
    if pd.isna(val):
        return np.nan
    try:
        r = float(val)
        return r if 1.0 <= r <= 10.0 else np.nan
    except (ValueError, TypeError):
        return np.nan
 
 
def _clean_genres(val):
    """מחזיר רשימת ז'אנרים נקייה."""
    if pd.isna(val):
        return []
    s = str(val).strip()
    if re.match(r'^(\\N|N/A|NA|\[\])$', s, re.IGNORECASE):
        return []
    try:
        parsed = ast.literal_eval(s)
        items  = [str(g).strip() for g in parsed] \
                 if isinstance(parsed, list) else [str(parsed).strip()]
    except (ValueError, SyntaxError):
        items = re.split(r'[,|;]', s)
    cleaned = []
    for item in items:
        g = item.strip().title()
        if g and re.match(r'^[A-Za-z\- ]{2,30}$', g):
            if not re.match(r'^(N/A|\\N|Nan|None|Na)$', g, re.IGNORECASE):
                cleaned.append(g)
    return cleaned
 
 
def _clean_actors(val):
    """מחלץ מזהי nm תקינים בלבד."""
    if pd.isna(val):
        return []
    s = str(val).strip()
    if re.match(r'^(\\N|\[\]|N/A)$', s, re.IGNORECASE):
        return []
    return re.findall(r'nm\d{7,8}', s)
 
 
def _clean_director(val):
    """מסיר sentinel בלבד, מחזיר כמו שהוא."""
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    if re.match(r'^(\\N|N/A|NA|nan|none)$', s, re.IGNORECASE):
        return np.nan
    return s

## 4. Feature Engineering


In [9]:
import pandas as pd
import numpy as np
import ast
import bisect

def _safe_list(val):
    if isinstance(val, list): return val
    if pd.isna(val): return []
    try:
        parsed = ast.literal_eval(str(val))
        return parsed if isinstance(parsed, list) else [parsed]
    except Exception:
        return [x.strip() for x in str(val).split(',') if x.strip()]

def add_static_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['num_genres']  = df['genres'].apply(lambda g: len(_safe_list(g)))
    df['is_too_short'] = (df['runtimeMinutes'] < 75).astype(float)
    df['is_too_long']  = (df['runtimeMinutes'] > 150).astype(float)
    return df

def calculate_historical_features(df: pd.DataFrame) -> pd.DataFrame:
    print("⏳ מחשב פיצ'רים היסטוריים בצורה מרוכזת (זה ייקח כדקה אחת בלבד)...")
    df = df.copy()
    
    df['_first_genre'] = df['genres'].apply(lambda g: _safe_list(g)[0] if _safe_list(g) else 'Unknown')
    genre_medians = df.groupby('_first_genre')['runtimeMinutes'].transform('median')
    df['relative_runtime_by_genre'] = df['runtimeMinutes'] / genre_medians
    df['relative_runtime_by_genre'] = df['relative_runtime_by_genre'].replace([np.inf, -np.inf], np.nan)
    
    df = df.sort_values('startYear').reset_index(drop=True)
    
    dir_history = {}
    dir_genre_history = {}
    actor_history = {}
    
    dir_past_avg, is_new_dir = [], []
    dir_genre_past_avg, dir_genre_count, is_new_dir_genre = [], [], []
    avg_actors_exp, is_new_actors = [], []
    
    global_mean = df['averageRating'].mean()
    genre_means = df.groupby('_first_genre')['averageRating'].mean().to_dict()
    
    for _, row in df.iterrows():
        year = row['startYear']
        rating = row['averageRating']
        genre = row['_first_genre']
        directors = _safe_list(row['directors'])
        actors = _safe_list(row['lead_actors_ids'])
        
        # במאי כללי
        movie_dir_ratings = []
        for d in directors:
            if d in dir_history:
                movie_dir_ratings.extend([r for y, r in dir_history[d] if y < year])
        if movie_dir_ratings:
            dir_past_avg.append(np.mean(movie_dir_ratings))
            is_new_dir.append(0.0)
        else:
            dir_past_avg.append(global_mean)
            is_new_dir.append(1.0)
            
        # במאי + ז'אנר
        movie_dir_genre_ratings = []
        for d in directors:
            key = (d, genre)
            if key in dir_genre_history:
                movie_dir_genre_ratings.extend([r for y, r in dir_genre_history[key] if y < year])
        if movie_dir_genre_ratings:
            dir_genre_past_avg.append(np.mean(movie_dir_genre_ratings))
            dir_genre_count.append(float(len(movie_dir_genre_ratings)))
            is_new_dir_genre.append(0.0)
        else:
            dir_genre_past_avg.append(genre_means.get(genre, global_mean))
            dir_genre_count.append(0.0)
            is_new_dir_genre.append(1.0)
            
        # שחקנים
        movie_actors_exp = 0
        for a in actors:
            if a in actor_history:
                movie_actors_exp += sum(1 for y in actor_history[a] if y < year)
        avg_actors_exp.append(float(movie_actors_exp))
        is_new_actors.append(1.0 if movie_actors_exp == 0 else 0.0)
        
        if not pd.isna(rating) and not pd.isna(year):
            for d in directors:
                dir_history.setdefault(d, []).append((year, rating))
                dir_genre_history.setdefault((d, genre), []).append((year, rating))
            for a in actors:
                actor_history.setdefault(a, []).append(year)
                
    df['director_past_avg'] = dir_past_avg
    df['is_new_director'] = is_new_dir
    df['director_genre_past_avg'] = dir_genre_past_avg
    df['director_genre_count'] = dir_genre_count
    df['is_new_director_genre'] = is_new_dir_genre
    df['avg_actors_past_experience'] = avg_actors_exp
    df['is_new_actors'] = is_new_actors
    
    return df.drop(columns=['_first_genre'])

## 4. prepare_data

In [10]:
def prepare_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # מימוש פונקציות העזר של הניקוי
    df['startYear']       = df['startYear'].apply(_clean_year)
    df['runtimeMinutes']  = df['runtimeMinutes'].apply(_clean_runtime)
    df['averageRating']   = df['averageRating'].apply(_clean_rating)
    df['genres']          = df['genres'].apply(_clean_genres)
    df['lead_actors_ids'] = df['lead_actors_ids'].apply(_clean_actors)
    df['directors']       = df['directors'].apply(_clean_director)

    # הוספת הפיצ'רים הפשוטים
    df = add_static_features(df)

    # ⚠️ חובה לנקות שורות ריקות לפני חישוב ההיסטוריה ⚠️
    df = df.dropna(subset=['averageRating', 'startYear']).reset_index(drop=True)

    # 🔥 הוספת השורה הזו: מחשב את כל הפיצ'רים ההיסטוריים במכה אחת מהירה! 🔥
    df = calculate_historical_features(df)

    # הסרת עמודות לא רלוונטיות
    COLS_TO_DROP = [
        'numVotes', 'BoxOffice', 'plot',
        'tconst', 'primaryTitle', 'Language', 'Country', 'budget',
    ]
    df = df.drop(columns=[c for c in COLS_TO_DROP if c in df.columns])

    return df

## 5. הכנת דאטת האימון



In [11]:
dataset_clean = prepare_data(dataset.copy())
dataset_clean = dataset_clean.dropna(subset=['averageRating']).reset_index(drop=True)

y = dataset_clean['averageRating'].values
X = dataset_clean.drop(columns=['averageRating'])

⏳ מחשב פיצ'רים היסטוריים בצורה מרוכזת (זה ייקח כדקה אחת בלבד)...


## 6. הגדרת Pipeline ומודלים

In [12]:
# ── 1. רשימת הפיצ'רים המספריים הסופיים (הכול כבר מחושב מראש) ──
FEATURE_COLS = [
    'startYear', 'runtimeMinutes', 'num_genres', 'is_too_short', 'is_too_long',
    'relative_runtime_by_genre', 'director_past_avg', 'avg_actors_past_experience',
    'is_new_director', 'is_new_actors', 'director_genre_past_avg', 'director_genre_count', 'is_new_director_genre'     
]

# ── 2. הגדרת ה-Preprocessor ─────────────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ]), FEATURE_COLS)
], remainder='drop')

# ── 3. הגדרת רשת פרמטרים מצומצמת ל-Random Forest למהירות ──────
param_grid_rf = {
    'max_depth': [15, None], 
    'min_samples_leaf': [2, 4]
}

# ── 4. ה-Pipelines הנקיים והמהירים ───────────────────────────
# ה-ElasticNet נעול על הפרמטרים הטובים ביותר - ירוץ ב-5 שניות!
pipeline_en = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model',        ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=2000, random_state=42)), 
])

pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model',        GridSearchCV(RandomForestRegressor(n_estimators=60, random_state=42, n_jobs=-1), 
                                  param_grid_rf, cv=2, scoring='neg_mean_squared_error', n_jobs=-1)),
])

## 7. לולאת CV ידנית

### למה לולאה ידנית?

בכל fold צריך לחשב את הפיצ'רים ההיסטוריים מחדש –
רק מנתוני ה-train של אותו fold.
sklearn לא מאפשר זאת בתוך `cross_val_predict` רגיל.

### מניעת Leakage בלולאה

```
fold i:
  df_tr = שורות האימון בלבד
  df_te = שורות הבדיקה בלבד

  _build_director_past_avg(df_tr, df_te)
  ← מחפש היסטוריה רק ב-df_tr
  ← df_te לא משפיע על החישוב
```

In [13]:
import time

kf = KFold(n_splits=10, shuffle=True, random_state=42)

def run_stability_cv(pipeline, X, y, model_name):
    print(f"\n🚀 מריץ 10-Fold CV מהיר ובדיקת יציבות עבור: {model_name}")
    print(f"==================================================")
    
    fold_metrics = []
    oof_preds = np.zeros(len(y))
    start_time = time.time()

    for fold_num, (train_idx, test_idx) in enumerate(kf.split(X)):
        current_fold = fold_num + 1
        
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx],      y[test_idx]

        # אימון המודל המהיר
        pipeline.fit(X_train, y_train)
        
        # חיזוי
        preds_train = pipeline.predict(X_train)
        preds_test  = pipeline.predict(X_test)
        oof_preds[test_idx] = preds_test

        # חישוב R²
        r2_train = r2_score(y_train, preds_train)
        r2_test  = r2_score(y_test, preds_test)
        gap      = r2_train - r2_test
        
        fold_metrics.append({
            'fold': current_fold, 'r2_train': r2_train, 'r2_test': r2_test, 'gap': gap
        })
        
        print(f"Fold {current_fold:02d}/10 | R² Train: {r2_train:.4f} | R² Test: {r2_test:.4f} | פער: {gap:.4f}")

    metrics_df = pd.DataFrame(fold_metrics)
    total_elapsed = time.time() - start_time
    
    print(f"\n── ✨ סיכום סופר-מהיר עבור: {model_name} ({total_elapsed:.1f} שניות) ──")
    print(f"  ממוצע R² אימון (Train): {metrics_df['r2_train'].mean():.4f} ± {metrics_df['r2_train'].std():.4f}")
    print(f"  ממוצע R² בחינה (Test):  {metrics_df['r2_test'].mean():.4f} ± {metrics_df['r2_test'].std():.4f}")
    print(f"  הפער הממוצע (Train - Test): {metrics_df['gap'].mean():.4f}")
    print(f"--------------------------------------------------\n")
    
    return oof_preds, metrics_df

# הרצה בפועל
oof_en, stability_en = run_stability_cv(pipeline_en, X, y, 'Elastic Net')
oof_rf, stability_rf = run_stability_cv(pipeline_rf, X, y, 'Random Forest')


🚀 מריץ 10-Fold CV מהיר ובדיקת יציבות עבור: Elastic Net
Fold 01/10 | R² Train: 0.2489 | R² Test: 0.2345 | פער: 0.0144
Fold 02/10 | R² Train: 0.2471 | R² Test: 0.2515 | פער: -0.0044
Fold 03/10 | R² Train: 0.2475 | R² Test: 0.2468 | פער: 0.0006
Fold 04/10 | R² Train: 0.2473 | R² Test: 0.2495 | פער: -0.0022
Fold 05/10 | R² Train: 0.2472 | R² Test: 0.2499 | פער: -0.0027
Fold 06/10 | R² Train: 0.2480 | R² Test: 0.2429 | פער: 0.0052
Fold 07/10 | R² Train: 0.2467 | R² Test: 0.2548 | פער: -0.0081
Fold 08/10 | R² Train: 0.2477 | R² Test: 0.2454 | פער: 0.0023
Fold 09/10 | R² Train: 0.2473 | R² Test: 0.2493 | פער: -0.0020
Fold 10/10 | R² Train: 0.2476 | R² Test: 0.2469 | פער: 0.0006

── ✨ סיכום סופר-מהיר עבור: Elastic Net (13.7 שניות) ──
  ממוצע R² אימון (Train): 0.2475 ± 0.0006
  ממוצע R² בחינה (Test):  0.2471 ± 0.0055
  הפער הממוצע (Train - Test): 0.0004
--------------------------------------------------


🚀 מריץ 10-Fold CV מהיר ובדיקת יציבות עבור: Random Forest
Fold 01/10 | R² Train: 0.4500 | 

## 8. הרצת CV

## 9. ניתוח חשיבות פיצ'רים

## 10. ניתוח שגיאות (Error Analysis)

## 11. ניתוח הוגנות (Fairness Analysis)

## 12. שמירת מודל סופי + prepare_data

### prepare_data

מקבלת רק את דאטת התחרות.
ניגשת ל-`df_train_global` שנשמר בזיכרון ה-notebook –
כך הפיצ'רים ההיסטוריים מחושבים על בסיס דאטת האימון.